# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.2: Integradores y Algoritmos de Integración

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/02_integradores_algoritmos.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Describir los algoritmos de integración más usados en DM: Verlet, Velocity-Verlet y Leap-Frog
- Implementar cada integrador en Python y comparar su desempeño
- Evaluar la conservación de energía y la estabilidad numérica de cada integrador
- Comprender la importancia del paso de tiempo en la precisión y estabilidad de la simulación
- Identificar el integrador más adecuado según el tipo de sistema

---

## 1. Instalación de Dependencias

In [ ]:
!pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

print("Bibliotecas importadas correctamente")

## 2. El Problema de la Integración Numérica

En DM necesitamos integrar las ecuaciones de movimiento de Newton:

$$\frac{d\mathbf{r}_i}{dt} = \mathbf{v}_i, \qquad \frac{d\mathbf{v}_i}{dt} = \frac{\mathbf{F}_i}{m_i}$$

Dado que no tienen solución analítica para sistemas de N cuerpos, se usan integradores numéricos.

Un buen integrador para DM debe ser:
- **Simpléctico**: conserva el volumen en el espacio de fases
- **Reversible en el tiempo**: $\mathbf{r}(t) \xrightarrow{\Delta t \to -\Delta t} \mathbf{r}(t-\Delta t)$
- **Estable**: errores acotados a largo plazo
- **Eficiente**: una evaluación de fuerzas por paso

## 3. Algoritmo de Verlet

Desarrollando en series de Taylor:

$$\mathbf{r}(t+\Delta t) = 2\mathbf{r}(t) - \mathbf{r}(t-\Delta t) + \frac{\mathbf{F}(t)}{m}\Delta t^2 + \mathcal{O}(\Delta t^4)$$

**Ventajas:** error global $\mathcal{O}(\Delta t^2)$, reversible en el tiempo, sin necesidad de calcular velocidades explícitamente.  
**Inconveniente:** necesita posiciones en dos pasos anteriores; las velocidades se calculan a posteriori.

In [ ]:
def oscilador_armonico_fuerza(x, k=1.0, m=1.0):
    """Fuerza de un oscilador armónico simple F = -kx"""
    return -k * x

def integrador_verlet(x0, v0, dt, n_pasos, k=1.0, m=1.0):
    """
    Integrador de Verlet para un oscilador armónico 1D.
    
    Args:
        x0: posición inicial
        v0: velocidad inicial
        dt: paso de tiempo
        n_pasos: número de pasos
        k: constante del resorte
        m: masa
    
    Returns:
        tiempos, posiciones, velocidades, energías
    """
    x_prev = x0 - v0 * dt  # posición ficticia en t-dt
    x_curr = x0
    
    posiciones = [x_curr]
    velocidades = [v0]
    tiempos = [0.0]
    
    for i in range(1, n_pasos):
        F = oscilador_armonico_fuerza(x_curr, k, m)
        a = F / m
        x_next = 2 * x_curr - x_prev + a * dt**2
        # Velocidad calculada entre pasos (orden inferior)
        v_curr = (x_next - x_prev) / (2 * dt)
        
        x_prev = x_curr
        x_curr = x_next
        
        posiciones.append(x_curr)
        velocidades.append(v_curr)
        tiempos.append(i * dt)
    
    pos = np.array(posiciones)
    vel = np.array(velocidades)
    Ek = 0.5 * m * vel**2
    Ep = 0.5 * k * pos**2
    return np.array(tiempos), pos, vel, Ek, Ep

t_v, x_v, v_v, Ek_v, Ep_v = integrador_verlet(x0=1.0, v0=0.0, dt=0.05, n_pasos=400)
print(f"Verlet | Deriva energía: {(Ek_v+Ep_v).std()/(Ek_v+Ep_v).mean():.4e}")

## 4. Algoritmo Velocity-Verlet

Versión equivalente pero con velocidades explícitas:

$$\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \frac{\mathbf{F}(t)}{2m}\Delta t^2$$

$$\mathbf{v}(t+\Delta t) = \mathbf{v}(t) + \frac{\mathbf{F}(t) + \mathbf{F}(t+\Delta t)}{2m}\Delta t$$

Es el integrador más utilizado en software moderno de DM por su precisión y fácil implementación de termostatos.

In [ ]:
def integrador_velocity_verlet(x0, v0, dt, n_pasos, k=1.0, m=1.0):
    """
    Integrador Velocity-Verlet para oscilador armónico 1D.
    """
    x, v = x0, v0
    posiciones, velocidades, tiempos = [x], [v], [0.0]
    
    for i in range(1, n_pasos):
        F_curr = oscilador_armonico_fuerza(x, k, m)
        a_curr = F_curr / m
        # Actualizar posición
        x_new = x + v * dt + 0.5 * a_curr * dt**2
        # Fuerza en la nueva posición
        F_new = oscilador_armonico_fuerza(x_new, k, m)
        a_new = F_new / m
        # Actualizar velocidad con promedio de aceleraciones
        v_new = v + 0.5 * (a_curr + a_new) * dt
        
        x, v = x_new, v_new
        posiciones.append(x)
        velocidades.append(v)
        tiempos.append(i * dt)
    
    pos = np.array(posiciones)
    vel = np.array(velocidades)
    Ek = 0.5 * m * vel**2
    Ep = 0.5 * k * pos**2
    return np.array(tiempos), pos, vel, Ek, Ep

t_vv, x_vv, v_vv, Ek_vv, Ep_vv = integrador_velocity_verlet(x0=1.0, v0=0.0, dt=0.05, n_pasos=400)
print(f"Velocity-Verlet | Deriva energía: {(Ek_vv+Ep_vv).std()/(Ek_vv+Ep_vv).mean():.4e}")

## 5. Algoritmo Leap-Frog

Variante del Velocity-Verlet que evalúa posiciones y velocidades en tiempos desfasados medio paso:

$$\mathbf{v}\left(t+\frac{\Delta t}{2}\right) = \mathbf{v}\left(t-\frac{\Delta t}{2}\right) + \frac{\mathbf{F}(t)}{m}\Delta t$$

$$\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}\left(t+\frac{\Delta t}{2}\right)\Delta t$$

Es el integrador por defecto en **GROMACS**. Equivalente al Velocity-Verlet pero organizado de forma diferente.

In [ ]:
def integrador_leapfrog(x0, v0, dt, n_pasos, k=1.0, m=1.0):
    """
    Integrador Leap-Frog para oscilador armónico 1D.
    """
    x = x0
    # Velocidad a medio paso negativo inicial
    F0 = oscilador_armonico_fuerza(x0, k, m)
    v_half = v0 - 0.5 * (F0 / m) * dt

    posiciones, velocidades, tiempos = [x0], [v0], [0.0]

    for i in range(1, n_pasos):
        F = oscilador_armonico_fuerza(x, k, m)
        v_half = v_half + (F / m) * dt        # v(t + dt/2)
        x = x + v_half * dt                   # r(t + dt)
        v_curr = v_half - 0.5 * (F / m) * dt  # v(t) aproximado

        posiciones.append(x)
        velocidades.append(v_curr)
        tiempos.append(i * dt)

    pos = np.array(posiciones)
    vel = np.array(velocidades)
    Ek = 0.5 * m * vel**2
    Ep = 0.5 * k * pos**2
    return np.array(tiempos), pos, vel, Ek, Ep

t_lf, x_lf, v_lf, Ek_lf, Ep_lf = integrador_leapfrog(x0=1.0, v0=0.0, dt=0.05, n_pasos=400)
print(f"Leap-Frog       | Deriva energía: {(Ek_lf+Ep_lf).std()/(Ek_lf+Ep_lf).mean():.4e}")

## 6. Comparación de los Tres Integradores

In [ ]:
# Solución analítica del oscilador armónico
omega = 1.0  # k=1, m=1
x_analitico = np.cos(omega * t_vv)

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Trayectorias
axes[0].plot(t_v,  x_v,  'b-',  label='Verlet',          linewidth=1.5, alpha=0.8)
axes[0].plot(t_vv, x_vv, 'g--', label='Velocity-Verlet', linewidth=1.5, alpha=0.8)
axes[0].plot(t_lf, x_lf, 'r:',  label='Leap-Frog',       linewidth=2.0, alpha=0.8)
axes[0].plot(t_vv, x_analitico, 'k-', label='Analítico', linewidth=1, alpha=0.5)
axes[0].set_xlabel('Tiempo', fontsize=12)
axes[0].set_ylabel('Posición', fontsize=12)
axes[0].set_title('Trayectorias del Oscilador Armónico', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energías totales
axes[1].plot(t_v,  Ek_v  + Ep_v,  'b-',  label='Verlet',          linewidth=1.5)
axes[1].plot(t_vv, Ek_vv + Ep_vv, 'g--', label='Velocity-Verlet', linewidth=1.5)
axes[1].plot(t_lf, Ek_lf + Ep_lf, 'r:',  label='Leap-Frog',       linewidth=2.0)
axes[1].set_xlabel('Tiempo', fontsize=12)
axes[1].set_ylabel('Energía Total', fontsize=12)
axes[1].set_title('Conservación de Energía Total', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Error vs solución analítica
axes[2].plot(t_v,  np.abs(x_v  - x_analitico), 'b-',  label='Verlet',          linewidth=1.5)
axes[2].plot(t_vv, np.abs(x_vv - x_analitico), 'g--', label='Velocity-Verlet', linewidth=1.5)
axes[2].plot(t_lf, np.abs(x_lf - x_analitico), 'r:',  label='Leap-Frog',       linewidth=2.0)
axes[2].set_xlabel('Tiempo', fontsize=12)
axes[2].set_ylabel('Error absoluto', fontsize=12)
axes[2].set_title('Error Acumulado vs Solución Analítica', fontsize=13)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_integradores.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Efecto del Paso de Tiempo (Δt)

El paso de tiempo es crítico: demasiado grande causa inestabilidad; demasiado pequeño hace la simulación ineficiente.

In [ ]:
# Analizar estabilidad en función del paso de tiempo
dts = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
errores = []
derivas = []

for dt_test in dts:
    n_p = int(20 / dt_test)  # simular hasta t=20
    try:
        t_t, x_t, _, Ek_t, Ep_t = integrador_velocity_verlet(x0=1.0, v0=0.0, dt=dt_test, n_pasos=n_p)
        x_analitico_t = np.cos(t_t)
        error = np.abs(x_t - x_analitico_t).max()
        E_total_t = Ek_t + Ep_t
        deriva = (E_total_t.max() - E_total_t.min()) / abs(E_total_t.mean())
    except Exception:
        error = np.nan
        deriva = np.nan
    errores.append(error)
    derivas.append(deriva)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].loglog(dts, errores, 'bo-', linewidth=2, markersize=8)
# Referencia de orden 2
dts_ref = np.array([0.01, 0.2])
axes[0].loglog(dts_ref, 0.5 * dts_ref**2, 'k--', label='O(Δt²)', alpha=0.7)
axes[0].set_xlabel('Paso de tiempo Δt', fontsize=12)
axes[0].set_ylabel('Error máximo en posición', fontsize=12)
axes[0].set_title('Error vs Δt (Velocity-Verlet)', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(dts, derivas, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Paso de tiempo Δt', fontsize=12)
axes[1].set_ylabel('Deriva de energía total', fontsize=12)
axes[1].set_title('Conservación de Energía vs Δt', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('efecto_dt.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nResumen:")
print(f"{'Δt':>6} | {'Error máximo':>14} | {'Deriva energía':>14}")
print("-" * 40)
for dt_v, err, der in zip(dts, errores, derivas):
    print(f"{dt_v:6.2f} | {err:14.4e} | {der:14.4e}")

## 8. Resumen Comparativo de Integradores

| Integrador | Orden de error | Velocidades explícitas | Uso en software | Notas |
|------------|---------------|------------------------|-----------------|-------|
| **Verlet** | O(Δt²) | No (a posteriori) | Histórico | Simple, reversible |
| **Velocity-Verlet** | O(Δt²) | Sí | NAMD, AMBER | Más estable, recomendado |
| **Leap-Frog** | O(Δt²) | Semidesfasadas | GROMACS | Eficiente para termostatos |
| **Runge-Kutta 4** | O(Δt⁴) | Sí | No común en DM | No simpléctico, 4 evaluaciones/paso |

> **Recomendación práctica:** usar **Velocity-Verlet** o **Leap-Frog** con Δt ≈ 1-2 fs para biomoléculas.

## 9. Ejercicios

### Ejercicio 1 (Básico)
Compara la trayectoria del oscilador armónico para los tres integradores con Δt = 0.5. ¿Cuál diverge primero?

### Ejercicio 2 (Intermedio)
Implementa el integrador de Runge-Kutta de orden 4 (RK4) para el mismo oscilador. Compara su error con el de Velocity-Verlet en función de Δt. ¿Por qué RK4 no se usa habitualmente en DM a pesar de su mayor precisión?

### Ejercicio 3 (Avanzado)
Extiende el integrador Velocity-Verlet a 2D para simular el movimiento orbital de un planeta alrededor del Sol (gravedad $\propto 1/r^2$). Verifica la conservación de la energía total y del momento angular.

In [ ]:
# Ejercicio 2 - Implementación de RK4
def integrador_rk4(x0, v0, dt, n_pasos, k=1.0, m=1.0):
    """
    Integrador Runge-Kutta de cuarto orden para el oscilador armónico.
    El estado es [x, v]. dy/dt = [v, -k*x/m]
    """
    def derivadas(t, y):
        x, v = y
        return [v, -k * x / m]

    y = np.array([x0, v0])
    posiciones, velocidades, tiempos = [x0], [v0], [0.0]

    for i in range(1, n_pasos):
        t_curr = (i - 1) * dt
        k1 = np.array(derivadas(t_curr, y))
        k2 = np.array(derivadas(t_curr + dt/2, y + dt/2 * k1))
        k3 = np.array(derivadas(t_curr + dt/2, y + dt/2 * k2))
        k4 = np.array(derivadas(t_curr + dt,   y + dt   * k3))
        y = y + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)

        posiciones.append(y[0])
        velocidades.append(y[1])
        tiempos.append(i * dt)

    return np.array(tiempos), np.array(posiciones), np.array(velocidades)

t_rk4, x_rk4, v_rk4 = integrador_rk4(x0=1.0, v0=0.0, dt=0.05, n_pasos=400)
x_analitico_rk4 = np.cos(t_rk4)

print(f"RK4             | Error máximo: {np.abs(x_rk4 - x_analitico_rk4).max():.4e}")
print(f"Velocity-Verlet | Error máximo: {np.abs(x_vv - x_analitico).max():.4e}")
print("\nNota: RK4 es más preciso, pero requiere 4 evaluaciones de fuerza por paso.")
print("Velocity-Verlet es simpléctico (conserva volumen en espacio de fases); RK4 no lo es.")

## 10. Recursos Adicionales

- **Libros:**
  - Frenkel & Smit, *Understanding Molecular Simulation*, Cap. 3-4 (2002)
  - Tuckerman, *Statistical Mechanics: Theory and Molecular Simulation*, Cap. 3 (2010)

- **Documentación de software:**
  - [GROMACS Manual - Integradores](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/md-algorithm.html)
  - [OpenMM - Integrators](https://openmm.org/documentation/latest/api-python/app.html#integrators)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Implementar los algoritmos Verlet, Velocity-Verlet y Leap-Frog en Python
- ✅ Comparar la precisión y estabilidad numérica de cada integrador
- ✅ Evaluar la conservación de energía como criterio de calidad de la simulación
- ✅ Seleccionar el paso de tiempo Δt adecuado para una simulación estable
- ✅ Identificar el integrador más adecuado según el ensamble estadístico

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.2: Integradores y Algoritmos**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.1-Fundamentos_de_DM-blue.svg)](01_fundamentos_dinamica_molecular.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.3_➡️-Condiciones_de_Contorno-green.svg)](03_condiciones_contorno_ensemble.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>